# Week 5: From Poems to Metadata — Reading Tabular Data


There is lots more we could do generating text and generating poetry. We could think about different ways of selecting words or phrases from our source text, we could work more on rhyming, and we could shape the code and our source text to explore different forms of creative expression.

However, we are going to shift direction and look at data about books, taking a more traditional data science approach to investigate different kinds of questions. The data set we will be working with is the full list of [Scotiabank Giller Prize](https://scotiabankgiller.com/) short (or long) list nominees for the past 30 years. This data includes book titles, authors, publication years, and where and when the authors were born,

New Python concepts:
- Reading a file line by line
- Comma-separated values (CSV) files and the `csv` module
- Data cleaning (next week)
- Writing a file (next week)
- Dictionaries

## 1. Our data set: the Giller Prize

The file `giller_nominees.csv` (in the same folder as this notebook) contains one row for every nominated book. For each book we have the year, title, and author, and for many of the authors, some biographical information: where they were born, their date of birth, where they live, and so on.

Like all real data sets, the data is not entirely consistent, there is some missing data, and it isn't always in the form that is useful for processing. Part of our task will be to clean the data.

Think about some questions you would like to ask about this data.

## 2. What is a CSV file?

A spreadsheet program such as Excel or Google Sheets displays data as a table of **rows** and **columns**. One of the simplest ways to store a spreadsheet in a file is the **comma-separated values** (CSV) format:

- Each line of the file is one **row** of the spreadsheet.
- Within a line, the values for each **column** are separated by commas.
- Usually the first line is a **header** that gives a name to each column.

Here are the first three lines of `giller_nominees.csv`, exactly as they appear in the file:

```
Year ,Title ,Author ,Publisher ,Publisher Location ,Born ,DOB,Residence ,DOD,Bio,Novel
1994,Funny Boy,Shyam Selvadurai,,,"Colombo, Sri Lanka",12-02-1965,"Toronto, ON",00/00/00,https://thecanadianencyclopedia.ca/...
1994,The Museum of Love,Steve Weiner,,,"Milwaukee, WI",1947,"London, England ",21/04/2024,https://en.wikipedia.org/wiki/Steve_Weiner,
```

Take a minute to look at these lines with a neighbour. What do you notice?

- Two commas in a row (`,,`) mean that a value is empty. Nobody has filled in the publisher yet.
- Some values, like `"Colombo, Sri Lanka"`, are wrapped in double quotes. Why might that be?
- Some of the column names have a space at the end (`Year ` rather than `Year`). We will come back to this issue later.

In class we will also open the file in JupyterLab and in a spreadsheet program to see the same data as a table.

## 3. Reading a file, line by line

In Week 4 we ran some code that opened `Shakespeare_Sonnets.txt` and promised we would explain it later. That time is now! Here is the same pattern applied to our new file.

To read the contents of a file and store it into a variable, we first need to *open* the file. The *open* operation connects the file to a Python variable. (This is the variable called `f` below.) There are a bunch of different ways to read the the contents of the file. For this course, we will focus on the method you see here that reads the contents of the file into a list variable where each element of the list is one line from the file.

The Python keyword `with` is also introduced here. This is a programming pattern that you should follow, but is not a core concept to deeply understand. (You will not be tested on this.)

The othter bit of arcane programming terminoloy is `encoding="utf-8"`. Characters have to be stored in computer memory as numbers (1s and 0s), so we need a way to encode each letter as a number. "utf-8" is an implementation of [Unicode](https://home.unicode.org/) which is an evolving standard for encoding as many different kinds of characters as possible (including emojis).

In [ ]:
# The "with" block makes sure the file is closed again when we are done with it.
# encoding="utf-8" tells Python how the characters in the file are stored.

with open("giller_nominees_clean.csv", encoding="utf-8") as f:
    # .readlines() reads the whole file and returns a list of strings,
    # one string per line of the file.
    lines = f.readlines()

print(len(lines), "lines in the file")

`lines` is a list, so everything we know about lists still works. Line 0 is the header, and each line after that is one row of the csv file, or one nominated book.

In [ ]:
print(lines[0])
print(lines[1])
print(lines[-1])

Notice the extra blank line after each one? Every line in a file ends with an invisible **newline character**, written `\n`, that marks where the line ends. We can see it if we look at the raw string without `print()`:

In [ ]:
lines[2]

The string method `.strip()` removes whitespace (spaces, tabs, and newlines) from the **beginning and end** of a string, but leaves the middle alone. We are going to make a lot of use of the `.strip()` method to clean up the data.

In [ ]:
line = lines[2].strip()
line

## 4. Splitting a row into columns

Last we we used `.split()` to split a line of poetry into a list of words. Today we will see how useful it can be!

Since the values in a row are separated by commas, it seems like `.split(",")` should break a row into a list of values. Let's try it on the third line, Steve Weiner's *The Museum of Love*.

In [ ]:
line = lines[2].strip()
values = line.split(",")
print(values)
print(len(values), "values")

This is great, but something looks a bit strange. Take a look at `'"Milwaukee'` and `' WI"'`. If we look at the file in a spreadsheet application like Google Sheets or Excel, "Milwaukee, WI" is in one column not too.  But `.split()` naively looks for commas whether or not they are within quotation marks. The same thing happen to many other cells like `"London, England "`.

This is exactly why those values were wrapped in double quotes. The quotes mean "the comma inside here is part of the value, not a separator between columns." But `.split(",")` doesn't know anything about quotes. It just cuts at every comma.

We *could* write our own code to handle quotes, but this problem is so common that there is a Python module to help us out.

In [ ]:
import csv

with open("giller_nominees.csv", encoding="utf-8") as f:
    reader = csv.reader(f)
    rows = list(reader)   # turn all of the rows into one big list

print(len(rows), "rows")
print(rows[2])
print(len(rows[2]), "values")

Now `"Milwaukee, WI"` stays together, and we get exactly 11 values, one per column. `rows` is a **list of lists**: each element of `rows` is itself a list representing one row.

To get the author of a book, we index twice: first to choose the row, then to choose the column.

In [ ]:
header = rows[0]
print(header)

book = rows[2]
print(book[1], "by", book[2], "born in", book[5])

## 5. Dictionaries

A Python **dictionary** works a lot like a real dictionary. In a real dictionary you look up a *word* to find its *definition*. In a Python dictionary you look up a **key** to find its **value**.

Python dictionaries share some similarities with Python lists. We look up a value in a Python list using it numeric index.

We write a dictionary with curly braces `{}`. (Remember, we write a list with square brackets `[]`.) Inside, each key is followed by a colon `:` and its value, and the key–value pairs are separated by commas.

Here is a small dictionary that maps the titles of some Giller winners to their authors.  This dictionary has three key-value pairs.  The keys are the book titles, and the values are the authors.  There are even handy dictionary methods so that we can get a *list* of the keys and a *list* of the values.

In [ ]:
winners = {
    "The Book of Secrets": "M.G. Vassanji",
    "A Fine Balance": "Rohinton Mistry",
    "Alias Grace": "Margaret Atwood",
}

print(winners)
print(type(winners))
print(winners.keys())
print(winners.values())

### Looking up a value

To look up a value we put the key in square brackets, just like indexing a list. With a list, we use a *position* number; with a dictionary, we use the *key*.

In [ ]:
winners["Alias Grace"]

What happens if we look up a key that isn't in the dictionary?

In [ ]:
winners["Fall on Your Knees"]

A `KeyError` means "there is no such key." We can check first with the `in` operator, which we met last week. With a dictionary, `in` checks the **keys**.

In [ ]:
title = "Fall on Your Knees"

if title in winners:
    print(f"{title} was written by {winners[title]}")
else:
    print(f"{title} is not in our dictionary")

### Adding and changing values

We add a new key–value pair, or change an existing one, with an assignment statement. If the key is already in the dictionary, its old value is replaced. If not, the pair is added.

In [ ]:
winners["How to Pronounce Knife"] = "Souvankham Thammavongsa"
print(winners)
print(len(winners), "entries")

**Keys are unique.** A dictionary can't contain the same key twice. Assigning to an existing key *replaces* its value. This is often exactly what we want, as we'll see when we start counting.



### Looping over a dictionary

A `for` loop over a dictionary visits each **key**. Then we can use the key to look up its value.

In [ ]:
for title in winners:
    print(f"{title} ({winners[title]})")

## 6. Each row as a dictionary: `csv.DictReader`

Let's return to the question of how to write `book["Born"]`. The `csv` module has a second tool, `csv.DictReader()`. It uses the header line as **keys** and turns every row into a dictionary.

We'll read the **clean** file we saved in section 6. (If you get a `FileNotFoundError`, go back and run the cells in section 6 first.)

In [ ]:
with open("giller_nominees_clean.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    nominees = list(reader)

print(len(nominees), "nominated books")   # notice the header is not counted as a book
print(nominees[1])

This makes it more legible to identify different aspects of the data we have about individual books.

In [ ]:
book = nominees[1]
print(book["Author"])
print(f"{book['Title']} by {book['Author']} ({book['Year']})")

Notice that inside the f-string we used single quotes around the key (`book['Title']`). That's because the f-string itself is surrounded by double quotes, and Python would get confused if we used double quotes on the inside too. 

Now we can start to take a look at this data in more interesting ways. Let's print all of the books from one year:

In [ ]:
# for each row of the table stored in nominees
for book in nominees:
    if book["Year"] == "1996":
        print(f"{book['Title']} by {book['Author']}")

Wait, shouldn't 1996 be an `int` rather than a `string`? 

The `csv` module reads every value as a *string*, even when it looks like a number. `book["Year"] == 1996` would be `False` for every book, because the string `"1996"` and the integer `1996` are not equal. If we ever need to do arithmetic with a year, we would have to convert it first using the function `int(book["Year"])`.

## 7. How many times was each author nominated?

Now we can answer our first real question. We want a new table with one row per **author**, giving the number of times they were nominated. A dictionary is perfect for this: the **keys** will be author names and the **values** will be counts.

This is one of the most useful patterns and is a variation on a pattern we have already seen, so let's go through it slowly:
- Start with an empty dictionary.
- For each book, look at its author.
    - If we've seen this author before, add 1 to their count.
    - If not, this is their first nomination, so set their count to 1.

In [ ]:
# create an empty dictionary
nomination_counts = {}

# Go through the rows in the table
for book in nominees:
    # set author to the author of the current row
    author = book["Author"]

    # if we have already started counting nominations for author
    # then we look up their nomination count so far and add one to it
    # otherwise, this is the first nomination we have see for this author
    # so we set its value in the dictionary to 1.
    if author in nomination_counts:
        nomination_counts[author] = nomination_counts[author] + 1
    else:
        nomination_counts[author] = 1

print(len(nomination_counts), "different authors")

In [ ]:
print(nomination_counts["Margaret Atwood"])
print(nomination_counts["Wayne Johnston"])